# 🔍 Extrato Atendimento Central — Gold vs Silver (Local)

Compares **local** `gold.extrato_atendimento_central` (`huntington_data_lake.duckdb`)
with **local** `silver.view_extrato_atendimentos_central` (`clinisys_all.duckdb`).

All heavy analysis runs **in DuckDB SQL** via `ATTACH` on an in-memory session.
Each cell opens a connection, runs its queries, and **closes it immediately** to avoid file locks.

### Sections
1. Setup & helpers
2. Schema Audit
3. Row Counts & Date Bounds
4. Key Overlap
5. Yearly Breakdown
6. Field-Level Discrepancies (matched rows)
7. Sample Mismatches per column
8. Sample: Only in Gold
9. Sample: Only in Silver


In [10]:
import duckdb, pandas as pd, re, warnings
warnings.filterwarnings('ignore')
GOLD_DB   = '../../database/huntington_data_lake.duckdb'
SILVER_DB = '../../database/clinisys_all.duckdb'

def open_con():
    c = duckdb.connect(':memory:')
    c.execute(f"ATTACH '{GOLD_DB}'   AS gold_db   (READ_ONLY)")
    c.execute(f"ATTACH '{SILVER_DB}' AS silver_db (READ_ONLY)")
    return c
# Quick connectivity test — connection is opened and closed immediately
con = open_con()
try:
    total_g = con.execute("SELECT count(*) FROM gold_db.gold.extrato_atendimento_central").fetchone()[0]
    total_s = con.execute("SELECT count(*) FROM silver_db.silver.view_extrato_atendimentos_central WHERE CAST(data AS DATE) >= '2019-01-01'").fetchone()[0]
    print(f'Gold rows:   {total_g:,}')
    print(f'Silver rows: {total_s:,}  (filtered >= 2019-01-01)')
finally:
    con.close()
    print('Connection closed.')


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Gold rows:   784,338
Silver rows: 499,994  (filtered >= 2019-01-01)
Connection closed.


## Part 1 — Schema Audit

Column names and types side-by-side.

In [11]:
import duckdb, pandas as pd, re, warnings
warnings.filterwarnings('ignore')
GOLD_DB   = '../../database/huntington_data_lake.duckdb'
SILVER_DB = '../../database/clinisys_all.duckdb'

def open_con():
    c = duckdb.connect(':memory:')
    c.execute(f"ATTACH '{GOLD_DB}'   AS gold_db   (READ_ONLY)")
    c.execute(f"ATTACH '{SILVER_DB}' AS silver_db (READ_ONLY)")
    return c
con = open_con()
try:
    gold_schema = con.execute("""
        SELECT column_name, data_type AS gold_type
        FROM duckdb_columns()
        WHERE database_name = 'gold_db'
          AND schema_name   = 'gold'
          AND table_name    = 'extrato_atendimento_central'
        ORDER BY column_index
    """).df()
    silver_schema = con.execute("""
        DESCRIBE silver_db.silver.view_extrato_atendimentos_central
    """).df()[['column_name','column_type']].rename(columns={'column_type':'silver_type'})
finally:
    con.close()

schema_cmp = gold_schema.merge(silver_schema, on='column_name', how='outer')
schema_cmp['match'] = schema_cmp.apply(
    lambda r: '✅' if r['gold_type'] == r['silver_type'] else
              ('➕ gold only' if pd.isna(r['silver_type']) else
               ('➕ silver only' if pd.isna(r['gold_type']) else '⚠️ type diff')), axis=1)
display(schema_cmp)


,column_name,gold_type,silver_type,match
0,agenda,INTEGER,INTEGER,✅
1,agenda_nome,VARCHAR,VARCHAR,✅
2,agendamento_id,INTEGER,INTEGER,✅
3,centro_custos,INTEGER,INTEGER,✅
4,centro_custos_nome,VARCHAR,VARCHAR,✅
5,chegou,VARCHAR,VARCHAR,✅
6,confirmado,INTEGER,INTEGER,✅
7,data,TIMESTAMP,TIMESTAMP,✅
8,data_agendamento_original,TIMESTAMP,TIMESTAMP,✅
9,evento,INTEGER,INTEGER,✅


## Part 2 — Row Counts & Date Bounds

In [12]:
import duckdb, pandas as pd, re, warnings
warnings.filterwarnings('ignore')
GOLD_DB   = '../../database/huntington_data_lake.duckdb'
SILVER_DB = '../../database/clinisys_all.duckdb'

def open_con():
    c = duckdb.connect(':memory:')
    c.execute(f"ATTACH '{GOLD_DB}'   AS gold_db   (READ_ONLY)")
    c.execute(f"ATTACH '{SILVER_DB}' AS silver_db (READ_ONLY)")
    return c
con = open_con()
try:
    gold_stats = con.execute("""
        SELECT count(*)                      AS total_rows,
               min(CAST(data AS DATE))       AS min_data,
               max(CAST(data AS DATE))       AS max_data,
               count(DISTINCT prontuario)    AS unique_prontuarios,
               count(DISTINCT agendamento_id) AS unique_agendamentos
        FROM gold_db.gold.extrato_atendimento_central
    """).df()
    silver_stats = con.execute("""
        SELECT count(*)                      AS total_rows,
               min(CAST(data AS DATE))       AS min_data,
               max(CAST(data AS DATE))       AS max_data,
               count(DISTINCT prontuario)    AS unique_prontuarios,
               count(DISTINCT agendamento_id) AS unique_agendamentos
        FROM silver_db.silver.view_extrato_atendimentos_central
        WHERE CAST(data AS DATE) >= '2019-01-01'
    """).df()
finally:
    con.close()

stats_cmp = pd.concat([
    gold_stats.assign(source='gold'),
    silver_stats.assign(source='silver (>= 2019-01-01)')
], ignore_index=True)[['source','total_rows','min_data','max_data',
                         'unique_prontuarios','unique_agendamentos']]
display(stats_cmp)


,source,total_rows,min_data,max_data,unique_prontuarios,unique_agendamentos
0,gold,784338,2019-01-01,2029-12-18,66070,784338
1,silver (>= 2019-01-01),499994,2019-01-01,2029-12-18,42547,499994


## Part 3 — Key Overlap

How many `agendamento_id` values are shared, exclusive to gold, or exclusive to silver?

In [13]:
import duckdb, pandas as pd, re, warnings
warnings.filterwarnings('ignore')
GOLD_DB   = '../../database/huntington_data_lake.duckdb'
SILVER_DB = '../../database/clinisys_all.duckdb'

def open_con():
    c = duckdb.connect(':memory:')
    c.execute(f"ATTACH '{GOLD_DB}'   AS gold_db   (READ_ONLY)")
    c.execute(f"ATTACH '{SILVER_DB}' AS silver_db (READ_ONLY)")
    return c
con = open_con()
try:
    overlap = con.execute("""
        WITH g AS (
            SELECT DISTINCT CAST(agendamento_id AS BIGINT) AS id
            FROM gold_db.gold.extrato_atendimento_central
        ),
        s AS (
            SELECT DISTINCT CAST(agendamento_id AS BIGINT) AS id
            FROM silver_db.silver.view_extrato_atendimentos_central
            WHERE CAST(data AS DATE) >= '2019-01-01'
        )
        SELECT 'In both (overlap)'  AS category, count(*) AS count
        FROM g JOIN s ON g.id = s.id
        UNION ALL
        SELECT 'Only in Gold',  count(*) FROM g WHERE id NOT IN (SELECT id FROM s)
        UNION ALL
        SELECT 'Only in Silver', count(*) FROM s WHERE id NOT IN (SELECT id FROM g)
    """).df()
finally:
    con.close()

display(overlap)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,category,count
0,In both (overlap),499994
1,Only in Gold,284344
2,Only in Silver,0


## Part 4 — Yearly Breakdown

In [14]:
import duckdb, pandas as pd, re, warnings
warnings.filterwarnings('ignore')
GOLD_DB   = '../../database/huntington_data_lake.duckdb'
SILVER_DB = '../../database/clinisys_all.duckdb'

def open_con():
    c = duckdb.connect(':memory:')
    c.execute(f"ATTACH '{GOLD_DB}'   AS gold_db   (READ_ONLY)")
    c.execute(f"ATTACH '{SILVER_DB}' AS silver_db (READ_ONLY)")
    return c
con = open_con()
try:
    yearly = con.execute("""
        WITH g AS (
            SELECT CAST(agendamento_id AS BIGINT) AS id,
                   YEAR(CAST(data AS DATE))       AS yr
            FROM gold_db.gold.extrato_atendimento_central
        ),
        s AS (
            SELECT CAST(agendamento_id AS BIGINT) AS id,
                   YEAR(CAST(data AS DATE))       AS yr
            FROM silver_db.silver.view_extrato_atendimentos_central
            WHERE CAST(data AS DATE) >= '2019-01-01'
        )
        SELECT
            COALESCE(g.yr, s.yr)                                        AS year,
            COUNT(DISTINCT g.id)                                         AS gold_total,
            COUNT(DISTINCT s.id)                                         AS silver_total,
            COUNT(DISTINCT CASE WHEN s.id IS NULL THEN g.id END)        AS only_in_gold,
            COUNT(DISTINCT CASE WHEN g.id IS NULL THEN s.id END)        AS only_in_silver
        FROM g FULL OUTER JOIN s ON g.id = s.id
        GROUP BY 1 ORDER BY 1
    """).df()
finally:
    con.close()

print('NOTE: only_in_gold = in gold but NOT in silver; only_in_silver = in silver but NOT in gold')
display(yearly)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

NOTE: only_in_gold = in gold but NOT in silver; only_in_silver = in silver but NOT in gold


,year,gold_total,silver_total,only_in_gold,only_in_silver
0,2019,49015,61,48954,0
1,2020,34267,427,33840,0
2,2021,92816,2757,90059,0
3,2022,121537,10949,110588,0
4,2023,128634,127734,900,0
5,2024,128382,128381,1,0
6,2025,134900,134900,0,0
7,2026,93399,93397,2,0
8,2027,1383,1383,0,0
9,2028,3,3,0,0


## Part 5 — Field-Level Discrepancies (Matched Rows)

Counts rows where gold and silver differ for each column, on the inner-joined set.
Strings are compared lowercased & accent-stripped; dates as `YYYY-MM-DD`; times as `HH:MM`.

In [15]:
import duckdb, pandas as pd, re, warnings
warnings.filterwarnings('ignore')
GOLD_DB   = '../../database/huntington_data_lake.duckdb'
SILVER_DB = '../../database/clinisys_all.duckdb'

def open_con():
    c = duckdb.connect(':memory:')
    c.execute(f"ATTACH '{GOLD_DB}'   AS gold_db   (READ_ONLY)")
    c.execute(f"ATTACH '{SILVER_DB}' AS silver_db (READ_ONLY)")
    return c
NORM_G = """
    SELECT
        CAST(agendamento_id AS BIGINT)                AS id,
        CAST(data AS DATE)                             AS data,
        STRFTIME(CAST(inicio AS TIMESTAMP), '%H:%M')   AS inicio,
        CAST(data_agendamento_original AS DATE)        AS data_ag_orig,
        CAST(medico AS BIGINT)                        AS medico,
        CAST(medico2 AS BIGINT)                       AS medico2,
        CAST(prontuario AS BIGINT)                    AS prontuario,
        CAST(evento AS BIGINT)                        AS evento,
        LOWER(STRIP_ACCENTS(TRIM(evento2)))            AS evento2,
        CAST(centro_custos AS BIGINT)                 AS centro_custos,
        CAST(agenda AS BIGINT)                        AS agenda,
        LOWER(STRIP_ACCENTS(TRIM(chegou)))             AS chegou,
        CAST(confirmado AS BIGINT)                    AS confirmado,
        CAST(paciente_codigo AS BIGINT)               AS paciente_codigo,
        LOWER(STRIP_ACCENTS(TRIM(paciente_nome)))      AS paciente_nome,
        LOWER(STRIP_ACCENTS(TRIM(medico_nome)))        AS medico_nome,
        LOWER(STRIP_ACCENTS(TRIM(medico_sobrenome)))   AS medico_sobrenome,
        LOWER(STRIP_ACCENTS(TRIM(medico2_nome)))       AS medico2_nome,
        LOWER(STRIP_ACCENTS(TRIM(centro_custos_nome))) AS centro_custos_nome,
        LOWER(STRIP_ACCENTS(TRIM(agenda_nome)))        AS agenda_nome,
        LOWER(STRIP_ACCENTS(TRIM(procedimento_nome)))  AS procedimento_nome
    FROM gold_db.gold.extrato_atendimento_central
"""
NORM_S = NORM_G.replace(
    'FROM gold_db.gold.extrato_atendimento_central',
    "FROM silver_db.silver.view_extrato_atendimentos_central\n    WHERE CAST(data AS DATE) >= '2019-01-01'"
)

DISC_COLS = [
    'data','inicio','data_ag_orig','medico','medico2','prontuario','evento','evento2',
    'centro_custos','agenda','chegou','confirmado','paciente_codigo',
    'paciente_nome','medico_nome','medico_sobrenome','medico2_nome',
    'centro_custos_nome','agenda_nome','procedimento_nome'
]
sums = ', '.join(f'SUM(({c!r} != \'matched\' AND (g.{c} IS DISTINCT FROM s.{c}))::INT) AS {c}' for c in DISC_COLS)

# Build the actual SQL more cleanly
disc_parts = [f'SUM((g.{c} IS DISTINCT FROM s.{c})::INT) AS {c}' for c in DISC_COLS]
disc_sql = f"""
    WITH g AS ({NORM_G}),
         s AS ({NORM_S}),
    matched AS (SELECT g.id, {', '.join(f'(g.{c} IS DISTINCT FROM s.{c}) AS d_{c}' for c in DISC_COLS)}
                FROM g JOIN s ON g.id = s.id)
    SELECT count(*) AS matched_rows,
           {', '.join(f'SUM(d_{c}::INT) AS {c}' for c in DISC_COLS)}
    FROM matched
"""

con = open_con()
try:
    disc_wide = con.execute(disc_sql).df()
finally:
    con.close()

matched_rows = int(disc_wide['matched_rows'].iloc[0])
disc = disc_wide.drop(columns='matched_rows').T.reset_index()
disc.columns = ['column', 'n_different']
disc['pct_different'] = (disc['n_different'] / matched_rows * 100).round(2)
disc = disc.sort_values('n_different', ascending=False).reset_index(drop=True)

print(f'Matched rows (inner join on agendamento_id): {matched_rows:,}')
print('\n--- Field Discrepancy Summary ---')
display(disc[disc['n_different'] > 0])


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Matched rows (inner join on agendamento_id): 499,994

--- Field Discrepancy Summary ---


,column,n_different,pct_different
0,paciente_nome,185.0,0.04
1,medico_nome,62.0,0.01
2,medico_sobrenome,62.0,0.01
3,data,2.0,0.00
4,procedimento_nome,1.0,0.00
5,confirmado,1.0,0.00
6,inicio,1.0,0.00
7,evento,1.0,0.00


## Part 6 — Sample Mismatches per Discrepant Column

In [16]:
import duckdb, pandas as pd, re, warnings
warnings.filterwarnings('ignore')
GOLD_DB   = '../../database/huntington_data_lake.duckdb'
SILVER_DB = '../../database/clinisys_all.duckdb'

def open_con():
    c = duckdb.connect(':memory:')
    c.execute(f"ATTACH '{GOLD_DB}'   AS gold_db   (READ_ONLY)")
    c.execute(f"ATTACH '{SILVER_DB}' AS silver_db (READ_ONLY)")
    return c
# Map normalised column alias -> raw SQL expression for gold and silver
COL_EXPRS = {
    'data':                     ('CAST(g.data AS DATE)',             'CAST(s.data AS DATE)'),
    'inicio':                   ("STRFTIME(CAST(g.inicio AS TIMESTAMP),'%H:%M')",
                                  "STRFTIME(CAST(s.inicio AS TIMESTAMP),'%H:%M')"),
    'data_ag_orig':             ('CAST(g.data_agendamento_original AS DATE)',
                                  'CAST(s.data_agendamento_original AS DATE)'),
    'medico':                   ('CAST(g.medico AS BIGINT)',         'CAST(s.medico AS BIGINT)'),
    'medico2':                  ('CAST(g.medico2 AS BIGINT)',        'CAST(s.medico2 AS BIGINT)'),
    'prontuario':               ('CAST(g.prontuario AS BIGINT)',     'CAST(s.prontuario AS BIGINT)'),
    'evento':                   ('CAST(g.evento AS BIGINT)',         'CAST(s.evento AS BIGINT)'),
    'evento2':                  ('g.evento2',                        's.evento2'),
    'centro_custos':            ('CAST(g.centro_custos AS BIGINT)',  'CAST(s.centro_custos AS BIGINT)'),
    'agenda':                   ('CAST(g.agenda AS BIGINT)',         'CAST(s.agenda AS BIGINT)'),
    'chegou':                   ('g.chegou',                         's.chegou'),
    'confirmado':               ('CAST(g.confirmado AS BIGINT)',     'CAST(s.confirmado AS BIGINT)'),
    'paciente_codigo':          ('CAST(g.paciente_codigo AS BIGINT)','CAST(s.paciente_codigo AS BIGINT)'),
    'paciente_nome':            ('g.paciente_nome',                  's.paciente_nome'),
    'medico_nome':              ('g.medico_nome',                    's.medico_nome'),
    'medico_sobrenome':         ('g.medico_sobrenome',               's.medico_sobrenome'),
    'medico2_nome':             ('g.medico2_nome',                   's.medico2_nome'),
    'centro_custos_nome':       ('g.centro_custos_nome',             's.centro_custos_nome'),
    'agenda_nome':              ('g.agenda_nome',                    's.agenda_nome'),
    'procedimento_nome':        ('g.procedimento_nome',              's.procedimento_nome'),
}

top_cols = disc[disc['n_different'] > 0].head(8)['column'].tolist()

for col in top_cols:
    if col not in COL_EXPRS:
        continue
    ge, se = COL_EXPRS[col]
    q = f"""
        SELECT CAST(g.agendamento_id AS BIGINT) AS agendamento_id,
               {ge} AS {col}_gold,
               {se} AS {col}_silver
        FROM gold_db.gold.extrato_atendimento_central g
        JOIN silver_db.silver.view_extrato_atendimentos_central s
          ON CAST(g.agendamento_id AS BIGINT) = CAST(s.agendamento_id AS BIGINT)
         AND CAST(s.data AS DATE) >= '2019-01-01'
        WHERE {ge} IS DISTINCT FROM {se}
        LIMIT 5
    """
    con = open_con()
    try:
        samples = con.execute(q).df()
    finally:
        con.close()
    n = int(disc.loc[disc.column == col, 'n_different'].iloc[0])
    print(f'\n--- [{col}]  {n:,} mismatched rows ---')
    display(samples)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


--- [paciente_nome]  185 mismatched rows ---


,agendamento_id,paciente_nome_gold,paciente_nome_silver
0,1278461,Monica Regina Bispo Zucatelli,Monica Regina Bispo dos Santos
1,1276763,Monica Regina Bispo Zucatelli,Monica Regina Bispo dos Santos
2,1276762,Monica Regina Bispo Zucatelli,Monica Regina Bispo dos Santos
3,1273867,Monica Regina Bispo Zucatelli,Monica Regina Bispo dos Santos
4,1272029,Monica Regina Bispo Zucatelli,Monica Regina Bispo dos Santos


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


--- [medico_nome]  62 mismatched rows ---


...


--- [inicio]  1 mismatched rows ---


,agendamento_id,inicio_gold,inicio_silver
0,1740266,13:00,11:00


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


--- [evento]  1 mismatched rows ---


,agendamento_id,evento_gold,evento_silver
0,1477956,765450,765446


## Part 7 — Sample Rows: Only in Gold

Appointments in `gold` with **no corresponding row** in silver.
Concentrated in 2019–2022 — pre-date full silver ingestion coverage.

In [ ]:
import duckdb, pandas as pd, re, warnings
warnings.filterwarnings('ignore')
GOLD_DB   = '../../database/huntington_data_lake.duckdb'
SILVER_DB = '../../database/clinisys_all.duckdb'

def open_con():
    c = duckdb.connect(':memory:')
    c.execute(f"ATTACH '{GOLD_DB}'   AS gold_db   (READ_ONLY)")
    c.execute(f"ATTACH '{SILVER_DB}' AS silver_db (READ_ONLY)")
    return c
con = open_con()
try:
    gold_only = con.execute("""
        SELECT CAST(g.agendamento_id AS BIGINT) AS agendamento_id,
               CAST(g.data AS DATE)              AS data,
               CAST(g.paciente_codigo AS BIGINT) AS paciente_codigo,
               CAST(g.medico AS BIGINT)          AS medico,
               CAST(g.evento AS BIGINT)          AS evento,
               g.centro_custos_nome,
               g.agenda_nome,
               g.procedimento_nome
        FROM gold_db.gold.extrato_atendimento_central g
        WHERE NOT EXISTS (
            SELECT 1
            FROM silver_db.silver.view_extrato_atendimentos_central s
            WHERE CAST(s.agendamento_id AS BIGINT) = CAST(g.agendamento_id AS BIGINT)
              AND CAST(s.data AS DATE) >= '2019-01-01'
        )
        ORDER BY CAST(g.data AS DATE) DESC
        LIMIT 30
    """).df()
finally:
    con.close()

print('Sample rows only in Gold (newest first; see Part 4 yearly table for totals):')
display(gold_only)


Sample rows only in Gold (oldest first; see Part 4 yearly table for totals):


,agendamento_id,data,paciente_codigo,medico,evento,centro_custos_nome,agenda_nome,procedimento_nome
0,1766429,2026-08-12,971721,<NA>,765503,7. HTT Brasília,Sala Coleta Seminal - BRASILIA,Espermograma
1,512998,2026-04-18,<NA>,1183,765180,6. HTT Belo Horizonte,Leci Amorim - BELO HORIZONTE,BLOQUEADO ***
2,784772,2024-09-30,<NA>,<NA>,71,None,None,None
3,817521,2023-11-06,790105,3554,76089,1. HTT SP - Ibirapuera,Eduardo Leme Alves da Motta - IBIRAPUERA,1ª Consulta Reprodução Humana - IB ***
4,784065,2023-09-26,<NA>,<NA>,71,None,None,None
5,650951,2023-09-14,<NA>,<NA>,71,None,None,None
6,793696,2023-08-25,177978,3765,79082,1. HTT SP - Ibirapuera,Guilherme Wood - IBIRAPUERA,Consulta por Telemedicina de Reprodução Humana...
7,793690,2023-08-25,177978,3765,79082,1. HTT SP - Ibirapuera,Thais Sanches Domingues - IBIRAPUERA,Consulta por Telemedicina de Reprodução Humana...
8,814873,2023-08-23,147165,<NA>,216,None,None,None
9,810026,2023-08-15,186526,<NA>,216,None,None,None


## Part 8 — Sample Rows: Only in Silver

Appointments in `silver` **not yet loaded into gold**.
Concentrated in 2025–2026 — gold table needs a refresh from the latest silver data.

In [18]:
import duckdb, pandas as pd, re, warnings
warnings.filterwarnings('ignore')
GOLD_DB   = '../../database/huntington_data_lake.duckdb'
SILVER_DB = '../../database/clinisys_all.duckdb'

def open_con():
    c = duckdb.connect(':memory:')
    c.execute(f"ATTACH '{GOLD_DB}'   AS gold_db   (READ_ONLY)")
    c.execute(f"ATTACH '{SILVER_DB}' AS silver_db (READ_ONLY)")
    return c
con = open_con()
try:
    silver_only = con.execute("""
        SELECT CAST(s.agendamento_id AS BIGINT) AS agendamento_id,
               CAST(s.data AS DATE)              AS data,
               CAST(s.paciente_codigo AS BIGINT) AS paciente_codigo,
               CAST(s.medico AS BIGINT)          AS medico,
               CAST(s.evento AS BIGINT)          AS evento,
               s.centro_custos_nome,
               s.agenda_nome,
               s.procedimento_nome
        FROM silver_db.silver.view_extrato_atendimentos_central s
        WHERE CAST(s.data AS DATE) >= '2019-01-01'
          AND NOT EXISTS (
            SELECT 1
            FROM gold_db.gold.extrato_atendimento_central g
            WHERE CAST(g.agendamento_id AS BIGINT) = CAST(s.agendamento_id AS BIGINT)
        )
        ORDER BY CAST(s.data AS DATE) DESC
        LIMIT 20
    """).df()
finally:
    con.close()

print('Sample rows only in Silver (most recent first; see Part 4 yearly table for totals):')
display(silver_only)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Sample rows only in Silver (most recent first; see Part 4 yearly table for totals):


,agendamento_id,data,paciente_codigo,medico,evento,centro_custos_nome,agenda_nome,procedimento_nome
